# Project 3 — Student Grade Manager

**Goal:** Develop a simple student gradebook that adds records, validates grades, computes averages, displays a report, and saves/loads CSV. This integrates the Basics syllabus.

**Level:** Basics · Project · **Study time:** 2–3 hours · **Prerequisite:** lessons 00–14

## You will learn

- Represent students with dictionaries and lists
- Validate numeric grades in the 0–100 range
- Create reusable reporting functions
- Round-trip data using the csv module

The runnable implementation is a worked example. Submit your own changes and regression tests in the independent section.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "basics", "projects"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. Model and validation

Represent the gradebook as a dictionary mapping a student name to a list of grades. Reject duplicate names, invalid grades, and non-finite floating-point values. Keep record operations in small functions.

In [ ]:
import math

def valid_grade(value):
    try:
        grade = float(value)
    except (TypeError, ValueError):
        return False
    return math.isfinite(grade) and 0 <= grade <= 100

def add_student(book, name):
    name = name.strip()
    if not name or name in book:
        raise ValueError("student name must be unique and nonempty")
    book[name] = []

def add_grade(book, name, grade):
    if name not in book:
        raise KeyError(name)
    if not valid_grade(grade):
        raise ValueError("grade must be a finite number from 0 to 100")
    book[name].append(float(grade))

**Check your understanding:** Why must NaN and infinity be rejected?

### 2. Reporting and sorting

The average is the sum divided by count; for no scores, use None to avoid implying the student earned zero. Format output only when displaying the report.

In [ ]:
def average(grades):
    return sum(grades) / len(grades) if grades else None

def report(book):
    return [(name, average(grades)) for name, grades in sorted(book.items())]

book = {}
add_student(book, "Ari")
add_student(book, "Bea")
add_grade(book, "Ari", 82)
add_grade(book, "Ari", 94)
add_grade(book, "Bea", 77)
for name, result in report(book):
    print(f"{name:<10} {result:.1f}" if result is not None else f"{name:<10} no grades")
assert average(book["Ari"]) == 88

**Check your understanding:** What happens if a student has no scores?

### 3. Persist to CSV and load again

Flatten each student-grade pair as one CSV row. A student without grades needs a separate blank-grade row to preserve the name. We use a temporary folder so running this cell never changes the repository.

In [ ]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

def save_book(book, path):
    with Path(path).open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "grade"])
        writer.writeheader()
        for name, grades in sorted(book.items()):
            for grade in (grades if grades else [""]):
                writer.writerow({"name": name, "grade": grade})

def load_book(path):
    result = {}
    with Path(path).open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            name, grade = row["name"].strip(), row["grade"].strip()
            if not name:
                raise ValueError("CSV contains an empty student name")
            result.setdefault(name, [])
            if grade:
                if not valid_grade(grade):
                    raise ValueError(f"Invalid grade for {name}")
                result[name].append(float(grade))
    return result

with TemporaryDirectory() as folder:
    filename = Path(folder) / "gradebook.csv"
    save_book(book, filename)
    reloaded = load_book(filename)
    print(reloaded)
    assert reloaded == book

**Check your understanding:** What should the loader do when it encounters malformed CSV headers?

### 4. Final project tests

Verify happy paths and failures. A project is not complete merely because one example runs: test invalid inputs, an empty gradebook, duplicate student names, and data round trips.

In [ ]:
assert report({}) == []
assert valid_grade("90")
assert not valid_grade("-1")
assert not valid_grade("nan")
assert not valid_grade("inf")
try:
    add_student(book, "Ari")
except ValueError:
    print("Duplicate correctly rejected")
try:
    add_grade(book, "Ari", 200)
except ValueError:
    print("Out-of-range score correctly rejected")
print("Project checks complete")

**Check your understanding:** What happens when a student name contains a comma? Will the csv module handle quoting?

## Independent project work

**Review evidence.** Test the sorted CSV export with names entered out of order and an empty dataset. Verify that exporting leaves the in-memory record order intact. Explain the decision behind your boundary case.

**Exercise 1.** Build a menu-driven interactive interface with add, list, and report commands.

In [ ]:
# TODO: Exercise 1
# Explain the steps, then implement and test your answer.

**Exercise 2.** Compute the highest student average while handling empty records.

In [ ]:
# TODO: Exercise 2
# Explain the steps, then implement and test your answer.

**Exercise 3.** Support deleting a student after confirmation.

In [ ]:
# TODO: Exercise 3
# Explain the steps, then implement and test your answer.

**Exercise 4.** Add a feature that exports a sorted report CSV without modifying the raw gradebook.

In [ ]:
# TODO: Exercise 4
# Explain the steps, then implement and test your answer.